# The Ermakov–Pinney Equation and Lie's Linearization Theorem

This tutorial demonstrates using **`symlie`** to analyze nonlinear ODEs, integrable Hamiltonian invariants, and point linearization theorems, based on **F. Güngör** (*Lie symmetry group methods for differential equations*, arXiv:1901.01543):

1. **The Ermakov–Pinney Equation**: $\ddot{y} + \omega^2(t) y = \frac{K}{y^3}$
   - 3-parameter $\mathfrak{sl}(2, \mathbb{R})$ symmetry algebra
   - Commutator table and the Lewis–Riesenfeld first integral
2. **Lie's Linearization Theorem for Second-Order ODEs**:
   - Tresse / Liouville invariants ($I_1 = f_{pppp} = 0$, $I_2 = 0$)
   - The Quadratic Liénard equation $\ddot{y} + \dot{y}^2 + A + B e^{-y} = 0$ and its linearization to $\ddot{Y} + A Y = -B$ via $Y = e^y$
3. **Maximal 8-Parameter Symmetry Algebras**: theory for $\ddot{y} + 3y\dot{y} + y^3 = 0$, contrasted with a restricted polynomial-ansatz calculation

In [ ]:
import sympy as sp

from symlie import (
    InfinitesimalGenerator,
    infinitesimals,
    lie_bracket,
    max_derivative_order,
    verify_generator,
)

sp.init_printing()

t = sp.symbols("t")
y = sp.Function("y")(t)
K = sp.symbols("K", positive=True)

# Autonomous Ermakov-Pinney equation: y'' = K / y^3
ep_eq = y.diff(t, 2) - K / y**3
print("ODE Order:", max_derivative_order(ep_eq, y, t))
sp.Eq(ep_eq, 0)

## 1. The 3-Dimensional $\mathfrak{sl}(2, \mathbb{R})$ Symmetry Algebra of Ermakov–Pinney

The Ermakov–Pinney equation admits the 3 symmetry generators:
- $\mathbf{v}_1 = \partial_t$
- $\mathbf{v}_2 = t \partial_t + \frac{1}{2} y \partial_y$
- $\mathbf{v}_3 = t^2 \partial_t + t y \partial_y$

In [ ]:
v1 = InfinitesimalGenerator(xi=(1,), phi=(0,))
v2 = InfinitesimalGenerator(xi=(t,), phi=(y / 2,))
v3 = InfinitesimalGenerator(xi=(t**2,), phi=(t * y,))

print("v_1 invariant:", verify_generator(ep_eq, y, t, v1))
print("v_2 invariant:", verify_generator(ep_eq, y, t, v2))
print("v_3 invariant:", verify_generator(ep_eq, y, t, v3))

# Commutator table
b12 = lie_bracket(v1, v2, y, t)
b13 = lie_bracket(v1, v3, y, t)
b23 = lie_bracket(v2, v3, y, t)

print("\nCommutation Relations:")
print("[v_1, v_2] =", b12)
print("[v_1, v_3] =", b13)
print("[v_2, v_3] =", b23)

assert b12.xi == (1,)
assert b13.xi == (2 * t,) and b13.phi == (y,)
assert b23.xi == (t**2,) and b23.phi == (t * y,)
print("Confirmed: Symmetry algebra is isomorphic to sl(2, R)!")

## 2. Invariant First Integral (The Lewis–Riesenfeld Invariant)

Using the symmetry generator $\mathbf{v}_3$, we construct the conserved first integral:
$$I = \frac{1}{2} \left( t \dot{y} - y \right)^2 + \frac{K t^2}{2 y^2}$$

We verify that $\frac{dI}{dt} \equiv 0$ on all solutions.

In [ ]:
I = sp.Rational(1, 2) * (t * y.diff(t) - y) ** 2 + (K * t**2) / (2 * y**2)

# Total time derivative of I
dI_dt = I.diff(t)
# Substitute the equation of motion y'' = K / y^3
dI_dt_on_shell = sp.simplify(dI_dt.subs(y.diff(t, 2), K / y**3))
print("dI/dt on solutions:", dI_dt_on_shell)
assert dI_dt_on_shell == 0
print("Verification: The Lewis-Riesenfeld invariant I is an exact first integral!")

## 3. Lie's Linearization Theorem and the Quadratic Liénard Equation

The quadratic Liénard equation:
$$\ddot{y} + \dot{y}^2 + A + B e^{-y} = 0$$
satisfies Lie's linearization criteria ($I_1 = f_{pppp} = 0, I_2 = 0$).

Under the change of dependent variable $Y(t) = e^{y(t)}$, the nonlinear ODE transforms to the linear harmonic oscillator:
$$\ddot{Y} + A Y = -B$$

In [ ]:
A_const, B_const = sp.symbols("A B")
lienard_ode = y.diff(t, 2) + y.diff(t) ** 2 + A_const + B_const * sp.exp(-y)

# Substitute y(t) = ln(Y(t))
Y = sp.Function("Y")(t)
transformed = sp.simplify(lienard_ode.subs(y, sp.log(Y)).doit())
print("Transformed ODE:")
display(transformed)

# Multiply by Y to get standard linear form
linear_form = sp.simplify(transformed * Y)
print("Linearized form (Y * ODE):")
display(linear_form)
assert sp.simplify(linear_form - (Y.diff(t, 2) + A_const * Y + B_const)) == 0
print("Verification: Quadratic Lienard equation linearizes to Y'' + A*Y = -B!")

## 4. Nonlinear Cubic ODE with Maximal 8-Parameter Symmetry

The nonlinear ODE:
$$\ddot{y} + 3y\dot{y} + y^3 = 0$$
belongs to the Chazy / Gambier classification and is linearizable to $\ddot{u} = 0$, admitting an 8-dimensional Lie algebra $\mathfrak{sl}(3, \mathbb{R})$. The calculation below deliberately searches only total-degree-1 polynomial coefficients, so it finds only the intersection of that full algebra with the chosen ansatz; it is not a derivation of all eight generators.

In [ ]:
cubic_ode = y.diff(t, 2) + 3 * y * y.diff(t) + y**3
print("Cubic ODE Order:", max_derivative_order(cubic_ode, y, t))

sol_cubic = infinitesimals(cubic_ode, y, t, ansatz_degree=1)
print(f"Dimension within degree-1 polynomial ansatz: {sol_cubic.ansatz_dimension}")
for i, gen in enumerate(sol_cubic.basis, 1):
    print(
        f"X_{i}: xi^t = {gen.xi[0]}, phi^y = {gen.phi[0]} | Valid: {verify_generator(cubic_ode, y, t, gen)}"
    )